# 01 — EDA, chọn novel class, chia tập
**Input:** BDD100K. **GPU:** không cần. **Output:** `art/splits/`. **GATE:** đặt `NOVEL` + `REASON` sau khi xem EDA.

In [ ]:
# === Setup (giống nhau ở mọi notebook) ===
SMOKE = True   # True: pipeline thật trên tập nhỏ. Run thật: False
SCENARIO = "missing"   # missing: base chưa có novel class | weak: base có ít ảnh novel
REPO_URL = "https://github.com/Le-Anh-Duy/topicx-yolo-expansion.git"
COMMIT = "main"   # run thật: pin commit hash
import os, subprocess, sys
R = "/tmp/repo"
if not os.path.exists(R):
    subprocess.run(["git", "clone", "-q", REPO_URL, R], check=True)
subprocess.run(["git", "-C", R, "fetch", "-q"], check=True)
subprocess.run(["git", "-C", R, "checkout", "-q", COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{R}/requirements.txt"], check=True)
sys.path.insert(0, f"{R}/src")
import numpy as np, pandas as pd
from topicx import common as C, pipeline as P
cfg = C.load_config(SMOKE, SCENARIO)
print(SCENARIO, cfg["art"], "| copy từ input:", C.sync_inputs(cfg), "file")

In [ ]:
STAGE = "splits"
# Test protocol (CPU, vài giây). Lỗi thì dừng notebook.
subprocess.run([sys.executable, "-m", "pytest", "-q", f"{R}/tests"], check=True)
C.env_report(cfg, STAGE);

## Health check dataset
Vài giây, không đọc cả file nhãn: đường dẫn, số ảnh, định dạng nhãn, ảnh mẫu có vẽ box. Có mục FAIL thì notebook dừng ở đây.

In [ ]:
hc, sample = P.health_check(cfg)
if sample is not None:
    display(sample.resize((640, 360)))   # box đỏ phải ôm đúng vật thể

In [ ]:
images, boxes, paths = P.load_bdd(cfg)
for name, t in P.eda(cfg, images, boxes).items():
    print(name); display(t)

In [ ]:
# GATE: đủ instance ở test, removal_cost thấp với base classes, pool có đủ positive
NOVEL = "bus" if SMOKE else None
REASON = "smoke" if SMOKE else ""
meta = P.prepare_splits(cfg, NOVEL, REASON, images, boxes, paths)
display(pd.DataFrame(meta["report"]).T); print("novel instances ở test:", meta["test_novel_instances"])